# CYTools landscape scans

A notebook-first path from database columns to cached derived geometry. Install with `pip install "cytools-workbench[notebook]"`. Requested 4D shards are downloaded on demand and cached; set `CYTOOLS_DB_DIR` when you want to use an explicit local snapshot.

In [ ]:
from cytools import quantities, quantity, scan, status, sweep

VERTEX_COUNTS = [5, 6, 7]

## Discover the schema

Database columns are zero-construction reads. Computed columns lazily build only the CYTools objects they need.

In [ ]:
quantities()

## Read database columns

The public Hodge numbers and filters use CYTools' N-lattice convention. `n` gives a reproducible bounded-memory stratified sample, not a globally uniform sample.

In [ ]:
basic = scan(
    ["h11", "h21", "chi", "n_vertices", "n_points"],
    n=100,
    n_vertices=VERTEX_COUNTS,
)
basic.head()

## Compute and cache geometry

`is_favorable` constructs a polytope; `n_intnums` continues through the ambient toric variety. Results are committed by `ks_id`, so rerunning this cell reuses the cache.

In [ ]:
derived = scan(
    ["h11", "is_favorable", "n_intnums"],
    n=100,
    n_vertices=VERTEX_COUNTS,
)
derived.head()

## Add a notebook quantity

Interactive quantities automatically run in this notebook process. Increase `version` whenever the quantity's algorithm or meaning changes.

In [ ]:
@quantity
def max_vertex_coordinate(g):
    """Largest absolute coordinate among the vertices."""
    return int(abs(g.polytope.vertices()).max())


custom = scan(
    ["h11", "max_vertex_coordinate"],
    n=100,
    n_vertices=VERTEX_COUNTS,
    version=1,
)
custom.head()

In [ ]:
status()

## Scale without collecting

Use `sweep` for long runs. It keeps memory bounded and returns counts instead of a DataFrame. The guard below makes the notebook safe to Run All.

In [ ]:
RUN_SWEEP = False
if RUN_SWEEP:
    summary = sweep(
        ["is_favorable", "n_intnums"],
        n=10_000,
        n_vertices=VERTEX_COUNTS,
    )
    print(summary)